# Профиль CSV кейса ML №18

Воспроизводимое описательное исследование локального `task/case2/smurf_players.csv`.
Одна строка — наблюдение игрока в матче. Это не обучение модели и не оценка качества детектора смурфов.

**Исполнение:** клетки последовательно выполнены CPython через `exec` в общем пространстве имён,
stdout фактически захвачен в outputs. Ядро Jupyter не запускалось.
Среда и статусы исполнения находятся в `notebook_execution.json`.


## 1. Источник, период и контроль неизменности

Путь разрешается относительно корня проекта. Можно выполнить notebook из корня, `night/` или
`night/analysis/`. Используется только стандартная библиотека Python. Никаких сетевых обращений,
кластерных запусков и изменений исходного CSV нет. SHA-256 фиксирует конкретную версию выборки.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import csv, hashlib, json, math, statistics

project_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'task/case2/smurf_players.csv').is_file()), None)
assert project_root is not None, 'Run inside the project directory'
source = project_root / 'task/case2/smurf_players.csv'
source_before = hashlib.sha256(source.read_bytes()).hexdigest()
with source.open(encoding='utf-8-sig', newline='') as stream:
    reader = csv.DictReader(stream)
    fieldnames = reader.fieldnames
    rows = list(reader)
assert rows and fieldnames
print('Источник:', source.relative_to(project_root))
print('SHA-256:', source_before)
print('Наблюдения:', len(rows), 'Столбцы:', len(fieldnames))
print('Безымянный первый столбец:', '' in fieldnames, '(индекс экспорта, не бизнес-ключ)')

Источник: task\case2\smurf_players.csv
SHA-256: 68da5dec95f23e55ebbb8a4cfd175063a415ab26e849ac1cdc05b477f68ef7a5
Наблюдения: 87 Столбцы: 46
Безымянный первый столбец: True (индекс экспорта, не бизнес-ключ)


## 2. Контракт и профилирование качества

Проверяем обязательные поля, уникальность ключей игрока в матче, форматы bool и числовые диапазоны.
Пустое значение `rank` нельзя заменять нулём или объявлять низким рангом.
Наличие всех положительных меток не означает, что происхождение разметки проверено.


In [2]:
required = ['match_id','player_slot','account_id','hero_id','kills','deaths','assists',
            'kda','gold_per_min','xp_per_min','hero_damage','last_hits','dur_min','win','smurf']
assert set(required) <= set(fieldnames)
assert all(r[k] != '' for r in rows for k in required)
assert all(r['win'] in {'True','False'} for r in rows)
assert all(r['is_radiant'] in {'True','False'} for r in rows)
assert all(float(r['dur_min']) > 0 for r in rows)
assert all(float(r[k]) >= 0 for r in rows for k in
           ['kills','deaths','assists','gold_per_min','xp_per_min','hero_damage','last_hits'])
duplicates_slot = len(rows) - len({(r['match_id'],r['player_slot']) for r in rows})
duplicates_account = len(rows) - len({(r['match_id'],r['account_id']) for r in rows})
assert duplicates_slot == 0 and duplicates_account == 0
missing_counts = {k:sum(r[k] == '' for r in rows) for k in fieldnames}
entirely_missing = [k for k,n in missing_counts.items() if n == len(rows)]
labels = Counter(r['smurf'] for r in rows)
print('Дубли (match_id, player_slot):', duplicates_slot)
print('Дубли (match_id, account_id):', duplicates_account)
print('Полностью пустые поля:', ', '.join(entirely_missing))
print('Метки smurf:', dict(labels))
print('PASS: контракт обязательных полей и уникальности')

Дубли (match_id, player_slot): 0
Дубли (match_id, account_id): 0
Полностью пустые поля: rank, calibration, rank_known, tier, kda_c, farm_pos, perf_z
Метки smurf: {'1': 87}
PASS: контракт обязательных полей и уникальности


## 3. Проверка производных признаков

Сверяем `kda = (kills + assists) / max(deaths, 1)`, `hd_pm = hero_damage / dur_min`
и `lh_pm = last_hits / dur_min`. Допуск абсолютного отклонения: 1e-6.
Проверка подтверждает арифметику файла, но не полноту и достоверность внешнего источника.


In [3]:
formula_errors = []
for index, r in enumerate(rows, start=2):
    expected = {
        'kda': (float(r['kills']) + float(r['assists'])) / max(float(r['deaths']),1),
        'hd_pm': float(r['hero_damage']) / float(r['dur_min']),
        'lh_pm': float(r['last_hits']) / float(r['dur_min']),
    }
    for name, calculated in expected.items():
        if not math.isfinite(float(r[name])) or abs(float(r[name])-calculated) > 1e-6:
            formula_errors.append({'csv_line':index,'field':name,'expected':calculated,'actual':r[name]})
assert not formula_errors, formula_errors
print('Проверенных формул:', len(rows)*3)
print('Отклонений:', len(formula_errors))
print('PASS: KDA, урон/мин и last hits/мин')

Проверенных формул: 261
Отклонений: 0
PASS: KDA, урон/мин и last hits/мин


## 4. Описательные показатели выборки

Винрейт рассчитывается как доля побед среди строк этого файла. Это не глобальный винрейт аккаунта
и не причинный эффект. Повторные наблюдения одного аккаунта не являются независимыми игроками.
Пороговые количества ниже служат только описанием и не применяются как классификатор.


In [4]:
accounts = sorted({r['account_id'] for r in rows})
matches = {r['match_id'] for r in rows}
wins = sum(r['win'] == 'True' for r in rows)
overall = {'observations':len(rows),'matches':len(matches),'accounts':len(accounts),
           'wins':wins,'win_rate_pct':100*wins/len(rows),
           'median_gpm':statistics.median(float(r['gold_per_min']) for r in rows),
           'median_xpm':statistics.median(float(r['xp_per_min']) for r in rows),
           'median_kda':statistics.median(float(r['kda']) for r in rows)}
print(json.dumps(overall, ensure_ascii=False, indent=2))
thresholds = {
    'KDA > 5':sum(float(r['kda']) > 5 for r in rows),
    'GPM > 700':sum(float(r['gold_per_min']) > 700 for r in rows),
    'Оба условия':sum(float(r['kda']) > 5 and float(r['gold_per_min']) > 700 for r in rows),
}
print('Описательные пороги:', json.dumps(thresholds, ensure_ascii=False))

{
  "observations": 87,
  "matches": 87,
  "accounts": 6,
  "wins": 77,
  "win_rate_pct": 88.50574712643679,
  "median_gpm": 711.0,
  "median_xpm": 1046.0,
  "median_kda": 9.333333333333334
}
Описательные пороги: {"KDA > 5": 66, "GPM > 700": 46, "Оба условия": 41}


## 5. Таблица шести аккаунтов

Для каждого аккаунта показываем число наблюдавшихся матчей, победы, винрейт в выбранном файле
и медианы GPM, XPM и KDA. Медианы аккаунтов и медианы всех строк имеют разные веса.


In [5]:
by_account = defaultdict(list)
for r in rows:
    by_account[r['account_id']].append(r)
account_table = []
for account, group in sorted(by_account.items()):
    account_wins = sum(r['win'] == 'True' for r in group)
    account_table.append({'account_id':account,'observations':len(group),
                          'matches':len({r['match_id'] for r in group}),
                          'wins':account_wins,'win_rate_pct':100*account_wins/len(group),
                          'median_gpm':statistics.median(float(r['gold_per_min']) for r in group),
                          'median_xpm':statistics.median(float(r['xp_per_min']) for r in group),
                          'median_kda':statistics.median(float(r['kda']) for r in group)})
assert len(account_table) == 6, 'Expected six accounts in this supplied file'
assert sum(r['observations'] for r in account_table) == len(rows)
assert sum(r['wins'] for r in account_table) == wins
print('| account_id | матчи | победы | винрейт, % | GPM медиана | XPM медиана | KDA медиана |')
print('|---|---:|---:|---:|---:|---:|---:|')
for r in account_table:
    print('| {} | {} | {} | {:.2f} | {:.1f} | {:.1f} | {:.3f} |'.format(
        r['account_id'],r['matches'],r['wins'],r['win_rate_pct'],
        r['median_gpm'],r['median_xpm'],r['median_kda']))

| account_id | матчи | победы | винрейт, % | GPM медиана | XPM медиана | KDA медиана |
|---|---:|---:|---:|---:|---:|---:|
| 307275024 | 10 | 9 | 90.00 | 802.5 | 1063.5 | 8.500 |
| 726454100 | 30 | 28 | 93.33 | 668.5 | 942.0 | 15.500 |
| 818013716 | 5 | 5 | 100.00 | 820.0 | 1142.0 | 13.250 |
| 826423518 | 9 | 9 | 100.00 | 793.0 | 1039.0 | 5.800 |
| 945403669 | 30 | 23 | 76.67 | 652.5 | 1078.0 | 5.464 |
| 970030081 | 3 | 3 | 100.00 | 957.0 | 1323.0 | 8.333 |


## 6. Независимая сверка с существующим профилем

Перерасчёт выполнен непосредственно из CSV. Следующая клетка только читает артефакты
`profile_smurf.py` и сверяет с ними результаты. Их наличие не требуется для первых пяти разделов.
При наличии расхождений утверждения остановят исполнение.


In [6]:
profile_path = project_root / 'night/analysis/smurf_profile.json'
assert profile_path.is_file(), 'Reproduce profile first: python night/analysis/profile_smurf.py'
profile = json.loads(profile_path.read_text(encoding='utf-8'))
assert profile['source_sha256'] == source_before
for key in ['observations','matches','accounts','wins','win_rate_pct']:
    assert math.isclose(profile['overall'][key], overall[key], abs_tol=1e-10)
assert profile['missing_counts'] == missing_counts
assert profile['label_counts'] == dict(labels)
assert profile['duplicate_player_match_keys'] == duplicates_slot
assert profile['duplicate_account_match_keys'] == duplicates_account
assert profile['derived_formula_errors'] == formula_errors
for r in account_table:
    existing = profile['per_account'][r['account_id']]
    assert r['observations'] == existing['observations']
    assert r['wins'] == existing['wins']
    for ours, theirs in [('median_gpm','gold_per_min'),('median_xpm','xp_per_min'),('median_kda','kda')]:
        assert math.isclose(r[ours], existing[theirs]['median'],abs_tol=1e-10)
assert hashlib.sha256(source.read_bytes()).hexdigest() == source_before
print('PASS: перерасчёт совпадает с smurf_profile.json')
print('PASS: исходный CSV не изменён')

PASS: перерасчёт совпадает с smurf_profile.json
PASS: исходный CSV не изменён


## 7. Выводы, ограничения и следующая проверка

В файле 87 наблюдений шести аккаунтов, 77 побед (88,51%). Медиана GPM равна 711.
Эти числа описывают выбранный файл и не оценивают распространённость смурфов.

- Все строки имеют `smurf=1`. Источник разметки не документирован, отрицательного класса и peer-контроля нет.
- `rank`, `calibration`, `rank_known`, `tier`, `kda_c`, `farm_pos`, `perf_z` полностью пусты.
- Нет временных меток матчей, режима и объективной позиции игрока, поэтому период и сопоставимость peers не подтверждены.
- Число матчей аккаунта в файле не равно размеру всей игровой истории и не доказывает «мало игр».
- Строки повторяются по аккаунтам, поэтому 87 строк нельзя считать 87 независимыми игроками.
- Isolation Forest не обучался. Accuracy, precision, recall, ROC-AUC и доля реальных смурфов не рассчитывались.

Для проверки ML-гипотезы нужен независимый набор обычных игроков и кандидатов, документированная
разметка, достаточная история на аккаунт, контекст роли и уровня матча, разбиение по аккаунтам и времени.
Порог аномальности следует выбирать на независимой валидации. Даже аномальный профиль сам по себе
не доказывает нарушение правил.
